# exp008 — ink_9um training corpus, group B (CPU-only Kaggle notebook)

**Settings:** **no GPU**, Internet **on**, notebook **private**. Config: `configs/exp008_corpus.yaml` (group B).
Builds the sparse 21-slice 9.6 um training layout (labels byte-exact, volume pooled with the official recipe) into
`/kaggle/working/corpus`, which becomes this notebook's private output and is attached to the later training notebooks.
Expected: ~22 GB download, ~35-50 min. Data come from the public open-data S3 bucket and the Hugging Face labels bucket. Numbers only, no images.

In [ ]:
%%bash
echo "cpus: $(nproc)"; free -g | sed -n 2p; df -h /kaggle/working | tail -1; python --version
mkdir -p /tmp/ks
git clone -q --depth 1 https://github.com/andreluizpedroso/Vesuvius-Challenge.git /tmp/ks/harness
git -C /tmp/ks/harness log -1 --format='harness %H'
true

In [ ]:
import os, yaml
cfg = yaml.safe_load(open("/tmp/ks/harness/configs/exp008_corpus.yaml"))
os.environ.update({"H": "/tmp/ks/harness", "CFG": "/tmp/ks/harness/configs/exp008_corpus.yaml", "GROUP": "B",
                    "CORPUS": "/kaggle/working/corpus", "PIP": " ".join(cfg["pip"])})
print({"group": os.environ["GROUP"], "pip": os.environ["PIP"]})

## 1. Dependencies (pinned): zarr, numcodecs, pyyaml (numpy is preinstalled)

In [ ]:
%%bash
pip install -q $PIP 2>&1 | tail -2
python -c "import zarr, numcodecs, numpy, yaml; print('zarr', zarr.__version__, '| numcodecs', numcodecs.__version__, '| numpy', numpy.__version__)"
true

## 2. Data terms
Already accepted by the owner for this account in exp000. Nothing here runs `vesuvius.accept_terms`.

## 3. Build the corpus (a failing segment is recorded in corpus.json; the others continue)

In [ ]:
%%bash
cd "$H/scripts"
python exp008_build_corpus.py --config "$CFG" --group "$GROUP" --out "$CORPUS" 2>&1 | grep -v -E "Unclosed|ResponseHandler|TCPConnector|ClientSession|^\s*$" | tail -80
true

## 4. Summary (numbers only)

In [ ]:
%%bash
echo "size: $(du -sh "$CORPUS" | cut -f1) | files: $(find "$CORPUS" -type f | wc -l)"
python3 - <<'EOF'
import json, os
c = json.load(open(os.environ["CORPUS"] + "/corpus.json"))
print("group", c["group"], "| ok", len(c["segments"]), "| FAILED", [f["key"] for f in c["failed"]], "| elapsed_s", c["elapsed_s"], "| fetched GB", round(c["fetch_stats"]["bytes"] / 1e9, 2), "| missing", c["fetch_stats"]["missing"])
for s in c["segments"]:
    print(f"  {s['key']:17s} shape {s['shape']} chunks fetched {s['volume_chunks_fetched']:5d} nonzero {s['volume_chunks_nonzero']:5d}  {s['seconds']:7.1f} s")
EOF
sha256sum "$CORPUS/corpus.json"
true